In [1]:
import pandas as pd
from river import datasets, base, compose, stats, stream, metrics, preprocessing, feature_extraction
import importlib
from geopy.distance import geodesic
from tqdm.auto import tqdm
from itertools import islice
import numpy as np
import matplotlib.pyplot as plt

import load_model
importlib.reload(load_model)

from load_model import ModelFactory

import geopandas as gpd
from shapely.geometry import Point
from collections import defaultdict, deque

In [2]:
class FeatureDistrict(base.Transformer):
    
    def __init__(self):
        nyc_boroughs = gpd.read_file("nybb_16b/nybb.shp")
        nyc_boroughs = nyc_boroughs.to_crs(epsg=4326)

        self.boroughs_list = [
            (row['geometry'], row['BoroName']) 
            for i, row in nyc_boroughs.iterrows()
        ]

    def which_dist(self, lat, lon):
        punkt = Point(lon, lat)
        
        for b_geom, b_name in self.boroughs_list:
            if b_geom.contains(punkt):
                return b_name
        return "Outside_NYC"

    def transform_one(self, x):
        x = x.copy()
        
        dt = x["pickup_datetime"]
        hour = dt.hour
        x["hour"] = hour
        x["day_of_week"] = dt.weekday()
        x["is_weekend"] = int(dt.weekday() >= 5)

        pickup_d = self.which_dist(x["pickup_latitude"], x["pickup_longitude"])
        dropoff_d = self.which_dist(x["dropoff_latitude"], x["dropoff_longitude"])

        x["pickup_district"] = pickup_d
        x["dropoff_district"] = dropoff_d

        x["within_district"] = int(
            pickup_d == dropoff_d and pickup_d != "Outside_NYC" and dropoff_d !="Outside_NYC"
        )
        a = (x["pickup_latitude"], x["pickup_longitude"])
        b = (x["dropoff_latitude"], x["dropoff_longitude"])
        dist = geodesic(a, b).kilometers
        x["distance"] = dist

        #x["store_and_fwd_flag"] = 1 if x["store_and_fwd_flag"] == "Y" else 0

        remove = ["vendor_id", "store_and_fwd_flag", "passenger_count","pickup_datetime"]
        for k in remove:
            x.pop(k, None)

        return x

In [3]:
factory = ModelFactory()
dataset = datasets.Taxis()
model_names = ['ARF', 'HTR', 'SRP']
pipelines = {
    name: factory.create(model_name=name, transformer=None)
    for name in model_names
}

transformer = FeatureDistrict()

mean_model = stats.Mean()

In [4]:
results = []

start = 500_000 
n = 5000

subset = islice(dataset, start, start + n)

models = defaultdict(lambda: {name: pipelines[name].clone() for name in pipelines})

for i, (x, y) in tqdm(enumerate(subset), total=n, desc="Streaming predictions"):

    row = {
        "ts": x["pickup_datetime"],
        "actual": y
    }

    x = transformer.transform_one(x)

    if x["within_district"] == 1:
        district = x["pickup_district"]
    else:
        district = "Global"

    row['district'] = district

    x.pop("pickup_district", None)
    x.pop("dropoff_district", None)

    y_pred_mean = mean_model.get()
    y_pred_mean = 0.0 if y_pred_mean is None else y_pred_mean

    mean_model.update(y)

    row["MEAN"] = y_pred_mean

    for name in pipelines:

        model = models[district][name]

        y_pred = model.predict_one(x)
        y_pred = y_pred if y_pred is not None else 0.0

        model.learn_one(x, y)

        row[name] = y_pred

    results.append(row)

Streaming predictions:   0%|          | 0/5000 [00:00<?, ?it/s]

In [11]:
res_df = pd.DataFrame(results)
res_df.head(10)

res_df.to_csv('taxi_diff_models_sample.csv')